# block-group-stack — ex1: build a ResNet BlockGroup from toy blocks

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `block-group-stack`. Running the final beacon cell reports progress against the `CNN: BlockGroup stack` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: BlockGroup stack` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`block-group-stack`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "block-group-stack"
DD_SUBTOPIC = "CNN: BlockGroup stack"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## ResNet BlockGroup stack — quick refresher

A `BlockGroup` is `n_blocks` `ResidualBlock`s wired in series, with the **first** block doing any downsampling/channel-change and the rest being identity-shaped:

```
Sequential(
    ResidualBlock(in_feats, out_feats, first_stride=first_stride),   # shape-changing
    *[ResidualBlock(out_feats, out_feats, first_stride=1) for _ in range(n_blocks - 1)],
)
```

**Two invariants of the pattern:**
1. **Only the first block changes shape.** Stride > 1 or `in_feats != out_feats` happens exactly once, at the top of the group. After that, every subsequent block is `out_feats → out_feats, stride=1`.
2. **Subsequent blocks get an identity skip.** Because `in_feats == out_feats` and stride == 1, the residual branch is a no-op — addition is well-defined without a projection.

**Why this matters.** It's the canonical way to compose a deep CNN: the *group* is the unit of width/resolution change, the *block* is the unit of additive refinement. ResNet-34 has 4 BlockGroups of (3, 4, 6, 3) blocks at widths (64, 128, 256, 512); ResNet-50/101/152 use the same group structure with different block counts.

**Stride convention.** The first-block stride is usually 1 for BlockGroup-0 (no downsample after the stem's MaxPool already cut resolution) and 2 for BlockGroups 1, 2, 3 — each later group halves the spatial resolution and doubles channels.

### Exercise 1 — build a ResNet BlockGroup from toy blocks

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the ResNet BlockGroup construction pattern — first block takes (in_feats, out_feats, first_stride), subsequent blocks take (out_feats, out_feats, stride=1) — using a toy `ResBlock` stand-in inside `nn.Sequential`.
> Keywords: resnet, block-group, sequential, first-block-stride
> ```

**KCs targeted:** `block-group-first-stride-trick`, `block-group-shape-invariant`

We're going to drill the BlockGroup *stacking* pattern, isolated from the (irrelevant for this skill) actual residual math. A toy `ResBlock` is provided:

```
class ResBlock(nn.Module):
    def __init__(self, in_feats, out_feats, first_stride=1):
        super().__init__()
        self.in_feats     = in_feats
        self.out_feats    = out_feats
        self.first_stride = first_stride
        # toy 1x1 conv that's just a learnable channel projection
        self.proj = nn.Conv2d(in_feats, out_feats, kernel_size=1, stride=first_stride)
    def forward(self, x):
        return self.proj(x)
```

(The provided cell defines this; you don't need to write it.)

Implement `ex1_make_block_group(in_feats, out_feats, n_blocks, first_stride)` that returns a `nn.Sequential` of `n_blocks` `ResBlock` instances:

1. **Block 0** = `ResBlock(in_feats, out_feats, first_stride=first_stride)`.
2. **Blocks 1..n_blocks-1** = `ResBlock(out_feats, out_feats, first_stride=1)`.

Wrap them in a single `nn.Sequential(*blocks)`.

**Why the asymmetry.** Only the first block changes the channel count (`in_feats → out_feats`) and applies any downsampling stride. After that, all subsequent blocks operate at the new resolution and channel count, with identity-shaped residual branches. This is the *only* pattern ResNet uses for block stacking — every CNN-family variant repeats it.

**Edge case.** When `n_blocks == 1`, the group is just the single shape-changing block (no extra identity-shaped blocks).

In [ ]:
import torch.nn as nn

class ResBlock(nn.Module):
    def __init__(self, in_feats, out_feats, first_stride=1):
        super().__init__()
        self.in_feats = in_feats
        self.out_feats = out_feats
        self.first_stride = first_stride
        self.proj = nn.Conv2d(in_feats, out_feats, kernel_size=1, stride=first_stride)
    def forward(self, x):
        return self.proj(x)


def ex1_make_block_group(in_feats: int, out_feats: int, n_blocks: int, first_stride: int):
    """Return a Sequential of n_blocks ResBlocks (first changes shape, rest are identity-shaped)."""
    raise NotImplementedError()


def _test_ex1():
    import torch.nn as nn

    # Canonical ResNet-34 stage-2 group: in=64, out=128, n_blocks=4, first_stride=2.
    group = ex1_make_block_group(in_feats=64, out_feats=128, n_blocks=4, first_stride=2)
    assert isinstance(group, nn.Sequential), 'must return nn.Sequential'
    assert len(group) == 4, f'expected 4 blocks, got {len(group)}'

    # Block 0: in_feats=64, out_feats=128, first_stride=2 (the only shape-changer).
    b0 = group[0]
    assert isinstance(b0, ResBlock)
    assert (b0.in_feats, b0.out_feats, b0.first_stride) == (64, 128, 2), (
        f'block 0 wrong: got {(b0.in_feats, b0.out_feats, b0.first_stride)}'
    )

    # Blocks 1..3: in_feats == out_feats == 128, first_stride == 1.
    for i in [1, 2, 3]:
        bi = group[i]
        assert (bi.in_feats, bi.out_feats, bi.first_stride) == (128, 128, 1), (
            f'block {i} wrong: got {(bi.in_feats, bi.out_feats, bi.first_stride)}'
        )

    # Forward smoke test — shape must downsample by first_stride and lift channels.
    x = t.randn(1, 64, 16, 16)
    y = group(x)
    # (1, 64, 16, 16) → block 0 (in=64,out=128,stride=2) → (1, 128, 8, 8)
    # → 3 more identity-shaped blocks → still (1, 128, 8, 8)
    assert y.shape == (1, 128, 8, 8), f'expected (1,128,8,8), got {tuple(y.shape)}'

    # Edge: n_blocks == 1 → group is just the shape-changer.
    group_one = ex1_make_block_group(32, 64, n_blocks=1, first_stride=2)
    assert len(group_one) == 1
    assert (group_one[0].in_feats, group_one[0].out_feats, group_one[0].first_stride) == (32, 64, 2)

    # Edge: first_stride == 1 + in == out (typical stage-0 group: no downsample).
    group_stage0 = ex1_make_block_group(64, 64, n_blocks=3, first_stride=1)
    assert len(group_stage0) == 3
    x0 = t.randn(1, 64, 56, 56)
    y0 = group_stage0(x0)
    assert y0.shape == (1, 64, 56, 56), 'stage-0 group should preserve shape exactly'

    # Confirm that EVERY block past index 0 has matching in/out (identity-shaped).
    for n_blocks in [2, 5, 7]:
        g = ex1_make_block_group(32, 96, n_blocks=n_blocks, first_stride=2)
        assert len(g) == n_blocks
        for i in range(1, n_blocks):
            assert g[i].in_feats == g[i].out_feats == 96, f'identity invariant broken at block {i}'
            assert g[i].first_stride == 1, f'stride invariant broken at block {i}'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_make_block_group(in_feats: int, out_feats: int, n_blocks: int, first_stride: int):
    import torch.nn as nn
    blocks = [ResBlock(in_feats, out_feats, first_stride=first_stride)]
    for _ in range(n_blocks - 1):
        blocks.append(ResBlock(out_feats, out_feats, first_stride=1))
    return nn.Sequential(*blocks)
```

**Why the splat `*blocks`.** `nn.Sequential` takes positional args for each module, not a list. `Sequential(blocks)` (no splat) would be wrong — Sequential would store the list as one element. `Sequential(*blocks)` unpacks so each ResBlock becomes its own ordered child.

**Equivalent one-liner.**
```
return nn.Sequential(
    ResBlock(in_feats, out_feats, first_stride),
    *[ResBlock(out_feats, out_feats, 1) for _ in range(n_blocks - 1)],
)
```
This is the exact form ARENA's official ResNet code uses.

**Why first_stride is a kwarg in the real ResBlock.** The *first* conv inside the residual branch has the stride; the skip branch has its own (1×1, stride=first_stride) projection when needed. Lifting the stride to the constructor lets the block decide where to apply it. Out of scope for this drill — our toy ResBlock just exposes the constructor signature.

**Why this composes with `module-composition`.** A ResNet is `Sequential(stem, *[BlockGroup(...) for _ in range(4)], head)`. BlockGroup is one level of the recursion — recognising the stacking pattern at this level makes the full assembly readable.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()